# El asesor de diseño en varios casos reales

`doekit.recommend_design` responde la pregunta *"¿qué método de diseño experimental
es el mejor para mi caso?"*. No es un AutoML mágico: es un **asesor transparente** que
combina dos capas —

1. **Reglas** (metodología clásica: Montgomery; handbook NIST/SEMATECH; guías de JMP)
   acotan el *shortlist* de métodos plausibles según el objetivo, el nº/tipo de factores
   y el orden del modelo.
2. **Evaluación** rankea ese shortlist con las métricas de `doekit` (D/A/G-eficiencia,
   varianza de predicción, nº de corridas) según tus **prioridades**, usando una
   **media geométrica ponderada**: un eje catastrófico (p. ej. predicción ≈ 0) *hunde*
   el score, sin que otro eje lo compense.

**Salvedad central:** "el mejor" es un **trade-off multiobjetivo** (pocas corridas vs
precisión de los coeficientes vs predicción en la región). Por eso el asesor **muestra
la tabla de alternativas** y expone sus supuestos — no esconde la decisión.

Este cuaderno recorre seis casos lado a lado.


In [1]:
import pandas as pd
import doekit as ed
from IPython.display import display

def mostrar(r, extra_caveats=False):
    print("RECOMENDADO:", r.method, " | escenario:", r.scenario)
    print(r.rationale)
    display(r.table)
    if extra_caveats:
        for c in r.caveats:
            print("  -", c)


## Caso 1 — Screening con presupuesto ajustado

Seis factores, solo 8 corridas disponibles: ¿cuáles influyen? El asesor debe elegir un
diseño de screening que quepa en el presupuesto.


In [2]:
r1 = ed.recommend_design("screening", factors=6, budget=8, seed=0)
mostrar(r1)

RECOMENDADO: Plackett-Burman  | escenario: {'goal': 'screening', 'n_factors': 6, 'budget': 8, 'model_order': 'linear'}
Para identificar los factores influyentes con 6 factores con un presupuesto de 8 corridas y un modelo 'linear', el mejor compromiso segun tus prioridades es Plackett-Burman (8 corridas, D-eficiencia 100.0%, G-eficiencia 100.0%).


,metodo,corridas,D_eff,G_eff,SPV_medio,en_presupuesto,soporta_modelo
0,Plackett-Burman,8,100.0,100.0,2.98,True,True
1,Definitive Screening,13,79.9,96.8,3.57,False,True


**Lectura:** gana **Plackett-Burman** (8 corridas, ortogonal). El Definitive Screening
existe pero no cabe en el presupuesto, así que queda listado pero descartado.


## Caso 2 — Superficie de respuesta (RSM), sin restricciones

Tres factores continuos, objetivo optimizar. Aquí compiten Box-Behnken, Central Composite,
Definitive Screening y un D-óptimo. Con prioridades equilibradas:


In [3]:
r2 = ed.recommend_design("optimization", factors=3, seed=0)
mostrar(r2)

RECOMENDADO: D-optimo  | escenario: {'goal': 'optimization', 'n_factors': 3, 'budget': None, 'model_order': 'quadratic'}
Para modelar la superficie de respuesta y localizar el optimo con 3 factores y un modelo 'quadratic', el mejor compromiso segun tus prioridades es D-optimo (13 corridas, D-eficiencia 46.1%, G-eficiencia 88.3%).


,metodo,corridas,D_eff,G_eff,SPV_medio,en_presupuesto,soporta_modelo
0,Box-Behnken,15,36.6,51.5,5.78,True,True
1,Central Composite,22,63.8,75.1,4.15,True,True
2,Definitive Screening,9,NaN,NaN,NaN,True,False
3,D-optimo,13,46.1,88.3,7.25,True,True


**Lectura:** el **D-óptimo** gana por economía de corridas y buena predicción; el DSD
sale como *no soporta el modelo* (no puede estimar el cuadrático completo con todas las
interacciones). Box-Behnken y Central Composite quedan a la vista como alternativas
clásicas robustas si prefieres una plantilla conocida.


## Caso 3 — El mismo caso, pero cambiando las prioridades

Aquí se ve el corazón del asesor: **"el mejor" depende de qué priorices**. Con el mismo
problema del Caso 2, movemos los pesos.


In [4]:
for etiqueta, prio in [("Priorizo POCAS CORRIDAS", {"runs":6, "precision":1, "prediction":1}),
                       ("Priorizo PRECISION",      {"runs":1, "precision":6, "prediction":1})]:
    r = ed.recommend_design("optimization", factors=3, priorities=prio, seed=0)
    print("%-26s -> %-18s (%d corridas)" % (etiqueta, r.method, r.design.n_runs))


Priorizo POCAS CORRIDAS    -> D-optimo           (13 corridas)
Priorizo PRECISION         -> Central Composite  (22 corridas)


**Lectura:** al priorizar pocas corridas gana un diseño chico (D-óptimo); al priorizar
precisión de los coeficientes gana el **Central Composite** (más corridas, mayor D-eficiencia).
Mismo problema, distinta respuesta — porque no hay un "mejor" absoluto.


## Caso 4 — Región restringida

Cuando parte del espacio experimental es inviable (una esquina prohibida, una relación
entre factores), los diseños-plantilla no aplican y el asesor fuerza el **diseño óptimo**
sobre candidatos factibles.


In [5]:
r4 = ed.recommend_design("optimization", factors=3, constrained=True, seed=0)
mostrar(r4, extra_caveats=True)

RECOMENDADO: D-optimo  | escenario: {'goal': 'optimization', 'n_factors': 3, 'budget': None, 'model_order': 'quadratic'}
Para modelar la superficie de respuesta y localizar el optimo con 3 factores y un modelo 'quadratic', el mejor compromiso segun tus prioridades es D-optimo (13 corridas, D-eficiencia 46.1%, G-eficiencia 88.3%).


,metodo,corridas,D_eff,G_eff,SPV_medio,en_presupuesto,soporta_modelo
0,D-optimo,13,46.1,88.3,7.25,True,True


  - Recomendacion condicional al supuesto model_order='quadratic' (y effect_size=1.0 para la potencia).
  - "El mejor" es un trade-off multiobjetivo (corridas vs precision vs prediccion): ajusta 'priorities' segun tu caso.
  - Fuera del catalogo actual: disenos de mezcla (simplex) y split-plot (factores dificiles de cambiar); si aplica tu caso, considera esos metodos por separado.


## Caso 5 — Presupuesto insuficiente

¿Qué pasa si el presupuesto no alcanza para ningún diseño que soporte el modelo? El asesor
**no falla en silencio**: lo señala como salvedad y sugiere el menor viable.


In [6]:
r5 = ed.recommend_design("optimization", factors=4, budget=6, seed=0)
mostrar(r5)
print("\nSalvedad principal:")
print(" ", r5.caveats[0])


RECOMENDADO: Definitive Screening  | escenario: {'goal': 'optimization', 'n_factors': 4, 'budget': 6, 'model_order': 'quadratic'}
Para modelar la superficie de respuesta y localizar el optimo con 4 factores con un presupuesto de 6 corridas y un modelo 'quadratic', el mejor compromiso segun tus prioridades es Definitive Screening (9 corridas).


,metodo,corridas,D_eff,G_eff,SPV_medio,en_presupuesto,soporta_modelo
0,Box-Behnken,27,25.2,31.6,10.78,False,True
1,Central Composite,32,76.2,100.0,5.27,False,True
2,Definitive Screening,9,NaN,NaN,NaN,False,False
3,D-optimo,18,44.8,59.4,13.06,False,True



Salvedad principal:
  Ningun diseno del catalogo cabe en el presupuesto (6) y soporta el modelo; se recomienda el menor viable (Definitive Screening, 9 corridas). Aumenta el presupuesto o reduce el modelo.


## Caso 6 — Factores categóricos

Los diseños RSM (Box-Behnken/CCD) asumen factores continuos. Con un factor categórico el
asesor evita esos métodos, prefiere factorial o D-óptimo, y **lo advierte**.


In [7]:
r6 = ed.recommend_design(
    "screening",
    factors=[ed.ContinuousFactor("x1", 0, 1), ed.CategoricalFactor("mat", ["A", "B", "C"])],
    seed=0)
mostrar(r6)
print("\nSalvedades relevantes:")
for c in r6.caveats:
    if "categor" in c.lower() or "mezcla" in c.lower():
        print(" -", c)


RECOMENDADO: D-optimo  | escenario: {'goal': 'screening', 'n_factors': 2, 'budget': None, 'model_order': 'linear'}
Para identificar los factores influyentes con 2 factores y un modelo 'linear', el mejor compromiso segun tus prioridades es D-optimo (7 corridas, D-eficiencia 41.6%, G-eficiencia 85.7%).


,metodo,corridas,D_eff,G_eff,SPV_medio,en_presupuesto,soporta_modelo
0,Factorial completo,6,NaN,NaN,NaN,True,False
1,D-optimo,7,41.6,85.7,3.5,True,True



Salvedades relevantes:
 - Fuera del catalogo actual: disenos de mezcla (simplex) y split-plot (factores dificiles de cambiar); si aplica tu caso, considera esos metodos por separado.
 - Hay factores categoricos: los disenos RSM (Box-Behnken/CCD) asumen factores continuos; para categoricos prefiere factorial o D-optimo.


## Resumen de los casos


In [8]:
casos = [("1. Screening 6f, budget 8", r1),
         ("2. RSM 3f (balance)", r2),
         ("4. Region restringida", r4),
         ("5. Budget insuficiente 4f", r5),
         ("6. Categorico", r6)]
resumen = pd.DataFrame([{"caso": n, "recomendado": r.method,
                         "corridas": r.design.n_runs} for n, r in casos])
display(resumen)

,caso,recomendado,corridas
0,"1. Screening 6f, budget 8",Plackett-Burman,8
1,2. RSM 3f (balance),D-optimo,13
2,4. Region restringida,D-optimo,13
3,5. Budget insuficiente 4f,Definitive Screening,9
4,6. Categorico,D-optimo,7


## Conclusión

El asesor **acota con reglas y decide con métricas**, siempre exponiendo las alternativas y
las salvedades. Su valor no es dar "la" respuesta, sino hacer explícito el trade-off y
señalar cuándo el caso se sale del catálogo (mezcla, split-plot) o del presupuesto.

Esta misma lógica alimenta la sección *"¿Fue el diseño apropiado?"* de cada reporte y será
el tool `recommend_design` del futuro servidor MCP.
